In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
#os.environ['OPENAI_API_KEY']

True

In [4]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1")

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [6]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_tck_db",  # Where to save data locally, remove if not necessary
)

In [7]:
from langchain_community.document_loaders import TextLoader

file_path = "tck.md"

loader = TextLoader(file_path, encoding="utf-8")

docs = loader.load()

assert len(docs) == 1
print(f"Toplam karakter sayısı: {len(docs[0].page_content)}")
print(f"Dosya içeriği özeti: {docs[0].page_content[:500]}...")

Toplam karakter sayısı: 252917
Dosya içeriği özeti: BİRİNCİ KİTAP

Genel Hükümler

BİRİNCİ KISIM

Temel İlkeler, Tanımlar ve Uygulama Alanı

BİRİNCİ BÖLÜM

Temel İlkeler ve Tanımlar

Ceza Kanununun amacı

Madde 1- (1) Ceza Kanununun amacı; kişi hak ve özgürlüklerini, kamu
düzen ve güvenliğini, hukuk devletini, kamu sağlığını ve çevreyi, toplum
barışını korumak, suç işlenmesini önlemektir. Kanunda, bu amacın
gerçekleştirilmesi için ceza sorumluluğunun temel esasları ile suçlar,
ceza ve güvenlik tedbirlerinin türleri düzenlenmiştir.

Suçta ve cezad...


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

chunks = text_splitter.split_documents(docs)

print(f"Split blog post into {len(chunks)} sub-documents.")

Split blog post into 320 sub-documents.


In [9]:
document_ids = vector_store.add_documents(documents=chunks)

print(document_ids[:3])

['aad97a0e-41ba-44cf-829e-27b02b8a0764', '5060683a-d41b-43c1-94ad-781f10bdee97', 'e41e7002-14a8-41b6-af50-8992db099c50']


In [10]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [11]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "Türk Ceza Kanunu kitabına erişimin var."
    "Soruları yanıtlarken bunu kullan."
)

agent = create_agent(model, tools, system_prompt=prompt)

In [12]:
query = ("TCK'ya göre kurumsal bir şirketle maaşımı başkaları ile paylaşamayacağıma dair bir sözleşme imzaladıysam ama maaşımı sızdırdıysam cezası nedir?")

In [13]:
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

TCK'ya göre kurumsal bir şirketle maaşımı başkaları ile paylaşamayacağıma dair bir sözleşme imzaladıysam ama maaşımı sızdırdıysam cezası nedir?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_8OgjXdWDdqStslSocXX3amJk)
 Call ID: call_8OgjXdWDdqStslSocXX3amJk
  Args:
    query: maaş bilgisinin paylaşılması veya sızdırılması TCK cezası
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'tck.md', 'start_index': 178909}
Content: Ticarî sır, bankacılık sırrı veya müşteri sırrı niteliğindeki bilgi veya
belgelerin açıklanması

Madde 239- (1) Sıfat veya görevi, meslek veya sanatı gereği vakıf olduğu
ticari sır, bankacılık sırrı veya müşteri sırrı niteliğindeki bilgi veya
belgeleri yetkisiz kişilere veren veya ifşa eden kişi, şikayet üzerine,
bir yıldan üç yıla kadar hapis v

In [ ]:
"""
================================== Ai Message ==================================

Türk Ceza Kanunu'na (TCK) göre bir şirketle maaşınızı başkaları ile paylaşmamanız gerektiğine dair bir sözleşme imzalamışsanız ve bu bilgiyi sızdırmışsanız, bu durum TCK'nın 239. maddesi kapsamında değerlendirilebilir.

TCK Madde 239'a göre:
- Sıfat veya görevi, meslek veya sanatı gereği vakıf olduğu ticari sır niteliğindeki bilgi veya belgeleri yetkisiz kişilere veren veya ifşa eden kişi, şikayet üzerine,
- Bir yıldan üç yıla kadar hapis ve beşbin güne kadar adlî para cezası ile cezalandırılır.

Eğer maaşınız, iş yeriniz tarafından "ticari sır" olarak ilan edilmiş ve sır saklama yükümlülüğü size yazılı olarak bildirilmiş ise, maaş bilgisini izinsiz paylaşmak bu kapsama girebilir.

Ancak, "maaş bilgisi" her zaman "ticari sır" sayılmayabilir. Ticari sır kabul edilip edilmeyeceği, işyerinin bu konuda açık politikası ve sözleşmede bunu nasıl düzenlediğiyle ilgilidir.

Sonuç olarak:
- Eğer maaşınız işyerinde açıkça ticari sır olarak tanımlanıp gizliliği yükümlülük altına alınmışsa ve siz bunu ifşa ettiyseniz, şikayet üzerine bir yıldan üç yıla kadar hapis ve adli para cezası alabilirsiniz.
- Eğer sadece disiplin ya da iş hukuku yönünden ihlal varsa (veya maaş ticari sır niteliğinde sayılmıyorsa), ceza TCK kapsamında olmaz ama iş akdiniz feshedilebilir veya tazminatsal yaptırımlar gündeme gelebilir.

Olayın bütün koşullarına göre değerlendirilmeli ve kesin bir hukuki görüş için bir avukata danışmanızda yarar vardır.
"""